# 01 · Data acquisition and schema

> Educational decision-support prototype trained on synthetic PaySim data. Outputs are risk scores and review priorities that help human investigators decide what to review first. This system makes no fraud or AML determination and performs no automatic blocking, account closure, customer risk rating, or regulatory reporting. Results on synthetic data do not establish real-world detection effectiveness, fairness, or regulatory suitability.

This notebook calls the `aml_triage` package and displays artifacts written to `reports/`. It defines no logic and reads every parameter from `configs/`. If the raw file has not been downloaded yet it explains the manual step and stops gracefully.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display

from aml_triage.config import load
from aml_triage.data.load import load_raw
from aml_triage.data.profiling import run_profile
from aml_triage.data.schema import load_schema, validate_frame

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load(ROOT / "configs" / "base.yaml")
schema = load_schema(ROOT / "configs" / "schema.yaml")
source = yaml.safe_load((ROOT / "configs" / "data_source.yaml").read_text())
print("config hash:", cfg.config_hash())

## Provenance and license (recorded in `configs/data_source.yaml`)

In [ ]:
display(pd.DataFrame({k: [v] for k, v in source.items()}).T.rename(columns={0: "value"}))

## Expected schema (confirmed against the real file by `validate-schema`, task V2)

In [ ]:
display(pd.DataFrame([{"column": c.name, "dtype": c.dtype, "role": c.role, "availability": c.availability, "description": c.description} for c in schema.columns]))

## Schema validation and profiling on the real file

Runs only when `paths.raw_csv` is set and the file exists.

In [ ]:
raw = cfg.paths.raw_csv
if raw is None or not (ROOT / raw).exists():
    display(Markdown("**Raw data not present.** Run `make data`, record the license in `configs/data_source.yaml`, then set `paths.raw_csv` in `configs/base.yaml` (tasks T017, T020)."))
else:
    df = load_raw(ROOT / raw, schema)
    report = validate_frame(df, schema)
    print(report.summary())
    assert report.ok, "schema validation failed"
    md_path, json_path = run_profile(df, schema, ROOT / cfg.paths.reports_dir, raw)
    display(Markdown((ROOT / cfg.paths.reports_dir / "data_quality.md").read_text()))

## Data dictionary

Generated by `python -m aml_triage data-dictionary`; shown here if present.

In [ ]:
dd = ROOT / cfg.paths.reports_dir / "data_dictionary.md"
display(Markdown(dd.read_text()) if dd.exists() else Markdown("_Data dictionary not generated yet._"))